In [13]:
# This file is created for fixing some of the issues that have arisen while working on Folium map.
# Most of these issues are caused by incompatability with 2021 version of the code(some changes to the libraries etc)

In [14]:
import folium
import wget
import pandas as pd

In [15]:
import folium
# Import folium MarkerCluster plugin
from folium.plugins import MarkerCluster
# Import folium MousePosition plugin
from folium.plugins import MousePosition
# Import folium DivIcon plugin
from folium.features import DivIcon

In [16]:
spacex_csv_file = wget.download('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv')
spacex_df=pd.read_csv(spacex_csv_file)

100% [................................................................................] 7710 / 7710

In [17]:
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]
launch_sites_df

,Launch Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,CCAFS SLC-40,28.563197,-80.576820
2,KSC LC-39A,28.573255,-80.646895
3,VAFB SLC-4E,34.632834,-120.610745


In [18]:
nasa_coordinate = [29.559684888503615, -95.0830971930759]
our_map = folium.Map(location=nasa_coordinate, zoom_start=8)

In [19]:
# Test to see if the libraries downloaded correctly
# Should open a map to Houston(Nasa Space Center) in a streetviewer mode
our_map

In [20]:
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]

In [21]:
# Function to assign color to launch outcome
def assign_marker_color(launch_outcome):
    if launch_outcome == 1:
        return 'red'
    else:
        return 'green'
    
spacex_df['marker_color'] = spacex_df['class'].apply(assign_marker_color)
spacex_df.tail(10)

,Launch Site,Lat,Long,class,marker_color
46,KSC LC-39A,28.573255,-80.646895,1,red
47,KSC LC-39A,28.573255,-80.646895,1,red
48,KSC LC-39A,28.573255,-80.646895,1,red
49,CCAFS SLC-40,28.563197,-80.576820,1,red
50,CCAFS SLC-40,28.563197,-80.576820,1,red
51,CCAFS SLC-40,28.563197,-80.576820,0,green
52,CCAFS SLC-40,28.563197,-80.576820,0,green
53,CCAFS SLC-40,28.563197,-80.576820,0,green
54,CCAFS SLC-40,28.563197,-80.576820,1,red
55,CCAFS SLC-40,28.563197,-80.576820,0,green


In [22]:
# The concept is the following - we want to group our launches by laucnh site, moreover each launch should be color-coded depending on the outcomes
# Color-coding is done using assign_marker_color function earlier.
# This cell groups by launch site and displays


marker_cluster = MarkerCluster()
our_map.add_child(marker_cluster)
for index, record in spacex_df.iterrows():
    val = record.values
    marker = folium.Marker([val[1], val[2]], icon=DivIcon(
        icon_size = (20, 20),
        icon_anchor = (0, 0),
        html = '<div style="font-size:12px;color:%s"><b>%s</b></div>' % (val[4], val[0])
    ))
    marker_cluster.add_child(marker)

# Be mindful of string-interpolation issues.

In [23]:
our_map

In [24]:
# Creating a function to calculate distance between 2 points on a map, using their coordinates
from math import sin, cos, sqrt, atan2, radians

def calculate_distance(lat1, lon1, lat2, lon2):
    # approximate radius of earth in km
    R = 6373.0

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    distance = R * c
    return distance

# Initial Map positioning
nasa_coords = [29.559684888503615, -95.0830971930759]
new_map = folium.Map(location=nasa_coordinate, zoom_start=8)

# adding mouse position(in top-right corner)
formatter = "function(num) {return L.Util.formatNum(num, 5);};"
mouse_position = MousePosition(
    position = 'topright',
    separator = ' Long: ',
    empty_string='NaN',
    lng_first=False,
    num_digits=20,
    prefix='Lat: ',
    lat_formatter=formatter,
    lng_formatter = formatter
)
new_map.add_child(mouse_position)


# Adding launching sites(with launches as clusters)
marker_cluster = MarkerCluster()
for index, row in spacex_df.iterrows():
    marker = folium.Marker([row[1], row[2]], icon=DivIcon(
        icon_size = (20, 20),
        icon_anchor = (0, 0),
        html = '<div style="font-size: 12; color:%s;"><b>%s</b></div>' % (row[4], row[0])
    ))
    marker_cluster.add_child(marker)
new_map.add_child(marker_cluster)

# Calculating distance between launch site and the nearest coastline
coast_coords = [28.5627, -80.56789]
launch_site_coords = [28.56213, -80.57725]
distance_coastline = calculate_distance(launch_site_coords[0], launch_site_coords[1], coast_coords[0], coast_coords[1])

#Adding a nearest coastline marker
distance_marker = folium.Marker(
    [coast_coords[0], coast_coords[1]],
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % "{:10.2f} KM".format(distance_coastline),
        )
    )
new_map.add_child(distance_marker)


# calculating distance to the closest railway
second_launch_coords = [28.57326, -80.64703]
second_closestrailway_coords = [28.57301, -80.65391]
second_distance = calculate_distance(second_launch_coords[0], second_launch_coords[1], second_closestrailway_coords[0], second_closestrailway_coords[1])

# Adding closest railway marker


def new_marker(num, coords):
    new_mark = folium.Marker(coords, icon=DivIcon(
        icon_size = (20, 20),
        icon_anchor = (0, 0),
        html = '<div style="font-size: 12; color: green;"><b>%s</b></div>' % "{:10.2f} KM".format(num)
    ))
    return new_mark

    
railway_marker = new_marker(second_distance, second_closestrailway_coords)
new_map.add_child(railway_marker)

# Calculating distance to the closest highway
highway_coords = [28.56260, -80.57067]
launch_coords = [28.56213, -80.57725]
distance_to_highway = calculate_distance(highway_coords[0], highway_coords[1], launch_coords[0], launch_coords[1])

# Adding a new marker(closest highway) to the map
highway_marker = new_marker(distance_to_highway, highway_coords)
new_map.add_child(highway_marker)

# calculating distance to the closest city
city_coords = [28.38751, -80.60532]
launch_coords = [28.56213, -80.57725]
distance_to_city = calculate_distance(city_coords[0], city_coords[1], launch_coords[0], launch_coords[1])

# Adding a new marker to the map
city_marker = new_marker(distance_to_city, city_coords)
new_map.add_child(city_marker)

# Drawing a line between launch_site and city
coordinates = [
    [city_coords[0],city_coords[1]],
    [launch_coords[0], launch_coords[1]]]
lines=folium.PolyLine(locations=coordinates, weight=1)
new_map.add_child(lines)

new_map

C:\Users\User\AppData\Local\Temp\ipykernel_6960\4275147745.py:44: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  marker = folium.Marker([row[1], row[2]], icon=DivIcon(
C:\Users\User\AppData\Local\Temp\ipykernel_6960\4275147745.py:47: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  html = '<div style="font-size: 12; color:%s;"><b>%s</b></div>' % (row[4], row[0])
